# V3: GAT + Mini-batch + HDBSCAN — Анализ результатов

Визуализация результатов эксперим��нтов 08–10:
1. **Эксп. 08** — статистика объединённого графа
2. **Эксп. 09** — кривые обучения GAT модели
3. **Эксп. 10** — оценка: in-domain, cross-domain, HDBSCAN кластеризация

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import torch

plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
UNIFIED_DIR = DATA_DIR / "graphs" / "v3_unified"
CROSS_DIR = DATA_DIR / "graphs" / "v3_cross"

---
## 1. Статистика объединённого графа (Эксп. 08)

In [ ]:
with open(UNIFIED_DIR / "stats.json") as f:
    stats = json.load(f)

print(f"Датасетов:       {stats['n_datasets']}")
print(f"Строк (row):     {stats['n_rows']:,}")
print(f"Токенов (token): {stats['n_tokens']:,}")
print(f"Рёбер:           {stats['n_edges']:,}")
print(f"Labeled pairs:   {stats['n_labeled']:,}")
print(f"  train:         {stats['n_train']:,} ({100*stats['n_train']/stats['n_labeled']:.1f}%)")
print(f"  val:           {stats['n_val']:,} ({100*stats['n_val']/stats['n_labeled']:.1f}%)")
print(f"  test:          {stats['n_test']:,} ({100*stats['n_test']/stats['n_labeled']:.1f}%)")
print(f"\nДатасеты: {', '.join(stats['datasets'])}")

In [ ]:
# Распределение split
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie: split proportions
split_sizes = [stats["n_train"], stats["n_val"], stats["n_test"]]
split_labels = [f"Train\n{stats['n_train']:,}", f"Val\n{stats['n_val']:,}", f"Test\n{stats['n_test']:,}"]
colors = ["#4CAF50", "#FF9800", "#F44336"]
axes[0].pie(split_sizes, labels=split_labels, colors=colors, autopct="%1.1f%%",
            startangle=90, textprops={"fontsize": 12})
axes[0].set_title("Распределение labeled pairs по split", fontsize=14)

# Bar: graph components
components = ["Rows", "Tokens", "Edges"]
values = [stats["n_rows"], stats["n_tokens"], stats["n_edges"]]
bars = axes[1].bar(components, values, color=["#2196F3", "#9C27B0", "#607D8B"])
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
                f"{val:,}", ha="center", fontsize=11, fontweight="bold")
axes[1].set_title("Компоненты объединённого графа", fontsize=14)
axes[1].set_ylabel("Количество")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "v3_graph_stats.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Эффект IDF-фильтрации и per-cell лимита на рёбра
fs = stats.get("filter_stats", {})
if fs:
    raw_e = fs["raw_edges"]
    removed_idf = fs["edges_removed_idf"]
    after_idf = fs["edges_after_idf"]
    removed_cell = fs["edges_removed_cell_limit"]
    final_e = fs["final_edges"]

    print(f"IDF порог:            {fs['idf_threshold']:.1%} ({int(fs['idf_threshold'] * stats['n_rows']):,} строк)")
    print(f"Токенов до / после:   {fs['raw_tokens']:,} → {fs['final_tokens']:,} "
          f"(удалено {fs['tokens_removed_idf']:,}, {100*fs['tokens_removed_idf']/fs['raw_tokens']:.1f}%)")
    print(f"Рёбер до фильтрации:  {raw_e:,}")
    print(f"  − IDF-фильтрация:   −{removed_idf:,} ({100*removed_idf/raw_e:.1f}%)")
    print(f"  = после IDF:         {after_idf:,}")
    print(f"  − per-cell лимит:    −{removed_cell:,} ({100*removed_cell/raw_e:.1f}%)")
    print(f"  = финальных рёбер:   {final_e:,} ({100*final_e/raw_e:.1f}% от исходных)")

    # Waterfall chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: waterfall — поэтапное уменьшение рёбер
    ax = axes[0]
    stages = ["Raw edges", "− IDF filter", "− Cell limit", "Final"]
    values = [raw_e, -removed_idf, -removed_cell, final_e]
    cumulative = [raw_e, after_idf, final_e, final_e]
    bottoms = [0, final_e + removed_cell, final_e, 0]

    colors_wf = ["#2196F3", "#F44336", "#FF9800", "#4CAF50"]
    bars = ax.bar(stages, [abs(v) for v in values], bottom=bottoms, color=colors_wf, edgecolor="white")

    for bar, val, cum in zip(bars, values, cumulative):
        y = bar.get_y() + bar.get_height() / 2
        label = f"{abs(val):,.0f}"
        if val < 0:
            label = f"−{abs(val):,.0f}\n({100*abs(val)/raw_e:.1f}%)"
        ax.text(bar.get_x() + bar.get_width()/2, y, label,
                ha="center", va="center", fontsize=10, fontweight="bold", color="white")

    ax.set_ylabel("Количество рёбер")
    ax.set_title("Waterfall: фильтрация рёбер графа", fontsize=14)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

    # Right: token filtering — before/after
    ax = axes[1]
    tok_labels = ["Токены\n(до)", "Удалено\nIDF", "Токены\n(после)"]
    tok_vals = [fs["raw_tokens"], fs["tokens_removed_idf"], fs["final_tokens"]]
    tok_colors = ["#2196F3", "#F44336", "#4CAF50"]
    bars = ax.bar(tok_labels, tok_vals, color=tok_colors)
    for bar, val in zip(bars, tok_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(tok_vals)*0.02,
                f"{val:,}", ha="center", fontsize=11, fontweight="bold")
    ax.set_ylabel("Количество токенов")
    ax.set_title(f"Фильтрация токенов (IDF > {fs['idf_threshold']:.0%})", fontsize=14)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "v3_filtering_stats.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("filter_stats отсутствует в stats.json — перезапустите эксперимент 08")

In [ ]:
# Распределение pos/neg в каждом split
train_pairs = torch.load(UNIFIED_DIR / "train_pairs.pt", weights_only=False)
val_pairs = torch.load(UNIFIED_DIR / "val_pairs.pt", weights_only=False)
test_pairs = torch.load(UNIFIED_DIR / "test_pairs.pt", weights_only=False)

splits = {"Train": train_pairs, "Val": val_pairs, "Test": test_pairs}
pos_counts = {name: int((p[:, 2] == 1).sum()) for name, p in splits.items()}
neg_counts = {name: int((p[:, 2] == 0).sum()) for name, p in splits.items()}

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(splits))
w = 0.35
bars1 = ax.bar(x - w/2, pos_counts.values(), w, label="Positive", color="#4CAF50")
bars2 = ax.bar(x + w/2, neg_counts.values(), w, label="Negative", color="#F44336")

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f"{int(bar.get_height()):,}", ha="center", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(splits.keys())
ax.set_ylabel("Количество пар")
ax.set_title("Positive / Negative пары по split")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "v3_split_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

for name, p in splits.items():
    pos = int((p[:, 2] == 1).sum())
    print(f"{name:6s}: {len(p):,} пар, {pos} pos ({100*pos/len(p):.1f}%), {len(p)-pos} neg")

---
## 2. Кривые обучения GAT модели (Эксп. 09)

In [ ]:
history_path = OUTPUT_DIR / "v3_gat_model.history.json"
with open(history_path) as f:
    history = json.load(f)

epochs = list(range(1, len(history["train_loss"]) + 1))
print(f"Эпох обучения: {len(epochs)}")
print(f"Финальный train_loss: {history['train_loss'][-1]:.4f}")
if history["val_loss"]:
    best_val_idx = int(np.argmin(history["val_loss"]))
    print(f"Лучший val_loss:     {history['val_loss'][best_val_idx]:.4f} (эпоха {best_val_idx + 1})")
if history.get("lr"):
    print(f"Финальный LR:        {history['lr'][-1]:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Loss curves ---
ax = axes[0]
ax.plot(epochs, history["train_loss"], label="Train Loss", color="#2196F3", linewidth=1.5)
if history["val_loss"]:
    val_epochs = list(range(1, len(history["val_loss"]) + 1))
    ax.plot(val_epochs, history["val_loss"], label="Val Loss", color="#F44336", linewidth=1.5)
    best_val_idx = int(np.argmin(history["val_loss"]))
    ax.axvline(best_val_idx + 1, color="green", linestyle="--", alpha=0.7,
               label=f"Best val @ ep {best_val_idx + 1}")
    ax.scatter([best_val_idx + 1], [history["val_loss"][best_val_idx]],
              color="green", s=80, zorder=5)

ax.set_xlabel("Эпоха")
ax.set_ylabel("Loss")
ax.set_title("Train / Val Loss")
ax.legend()

# --- Learning rate ---
ax = axes[1]
if history.get("lr"):
    lr_epochs = list(range(1, len(history["lr"]) + 1))
    ax.plot(lr_epochs, history["lr"], color="#9C27B0", linewidth=1.5)
    ax.set_xlabel("Эпоха")
    ax.set_ylabel("Learning Rate")
    ax.set_title("Learning Rate (ReduceLROnPlateau)")
    ax.set_yscale("log")
else:
    ax.text(0.5, 0.5, "LR history not available", transform=ax.transAxes, ha="center")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "v3_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Smoothed loss (скользящее среднее)
def smooth(values, window=10):
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

fig, ax = plt.subplots(figsize=(10, 5))

window = min(10, len(epochs) // 5) if len(epochs) > 20 else 3
sm_train = smooth(history["train_loss"], window)
sm_epochs = list(range(window, len(history["train_loss"]) + 1))

ax.plot(epochs, history["train_loss"], alpha=0.2, color="#2196F3")
ax.plot(sm_epochs, sm_train, label=f"Train (MA-{window})", color="#2196F3", linewidth=2)

if history["val_loss"]:
    sm_val = smooth(history["val_loss"], window)
    val_epochs_all = list(range(1, len(history["val_loss"]) + 1))
    sm_val_epochs = list(range(window, len(history["val_loss"]) + 1))
    ax.plot(val_epochs_all, history["val_loss"], alpha=0.2, color="#F44336")
    ax.plot(sm_val_epochs, sm_val, label=f"Val (MA-{window})", color="#F44336", linewidth=2)

ax.set_xlabel("Эпоха")
ax.set_ylabel("Loss")
ax.set_title(f"Сглаженные кривые обучения (окно={window})")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "v3_training_curves_smooth.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. Результаты оценки (Эксп. 10)

In [ ]:
with open(OUTPUT_DIR / "v3_evaluation_results.json") as f:
    results = json.load(f)

print("=" * 50)
print("IN-DOMAIN TEST")
print("=" * 50)
if results.get("in_domain"):
    for k, v in results["in_domain"].items():
        print(f"  {k:20s}: {v:.4f}")

print()
if results.get("in_domain_clustering"):
    print("In-domain HDBSCAN:")
    for k, v in results["in_domain_clustering"].items():
        if isinstance(v, float):
            print(f"  {k:20s}: {v:.4f}")
        else:
            print(f"  {k:20s}: {v}")

print()
print("=" * 50)
print("CROSS-DOMAIN")
print("=" * 50)
if results.get("cross_domain"):
    for r in results["cross_domain"]:
        auc = r.get("roc_auc", "N/A")
        ap = r.get("avg_precision", "N/A")
        auc_s = f"{auc:.4f}" if isinstance(auc, (int, float)) else auc
        ap_s = f"{ap:.4f}" if isinstance(ap, (int, float)) else ap
        print(f"  {r['name']:20s}  AUC={auc_s}  AP={ap_s}")
        if "clustering" in r:
            cl = r["clustering"]
            print(f"    {'':20s}  clusters={cl['n_clusters']}  coverage={cl['coverage']:.1%}  noise={cl['noise_ratio']:.1%}")

In [ ]:
# --- In-domain метрики ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar: ROC-AUC / AP
ax = axes[0]
if results.get("in_domain"):
    metrics = results["in_domain"]
    names = list(metrics.keys())
    vals = list(metrics.values())
    colors_bar = ["#2196F3", "#4CAF50"]
    bars = ax.bar(names, vals, color=colors_bar[:len(names)])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.4f}", ha="center", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_title("In-domain Test Metrics", fontsize=14)
    ax.set_ylabel("Score")

# Pie: HDBSCAN coverage
ax = axes[1]
if results.get("in_domain_clustering"):
    cl = results["in_domain_clustering"]
    clustered = cl["n_total"] - cl["n_noise"]
    noise = cl["n_noise"]
    ax.pie([clustered, noise],
           labels=[f"Clustered\n{clustered:,}", f"Noise\n{noise:,}"],
           colors=["#4CAF50", "#9E9E9E"],
           autopct="%1.1f%%", startangle=90, textprops={"fontsize": 12})
    title_parts = [f"HDBSCAN: {cl['n_clusters']} кластеров"]
    if "ari" in cl:
        title_parts.append(f"ARI={cl['ari']:.3f}")
    if "nmi" in cl:
        title_parts.append(f"NMI={cl['nmi']:.3f}")
    ax.set_title(", ".join(title_parts), fontsize=13)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "v3_in_domain_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Cross-domain сравнение ---
cross = results.get("cross_domain", [])
if cross:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    names = [r["name"] for r in cross]
    aucs = [r.get("roc_auc", 0) for r in cross]
    aps = [r.get("avg_precision", 0) for r in cross]

    # ROC-AUC / AP bar chart
    ax = axes[0]
    x = np.arange(len(names))
    w = 0.35
    b1 = ax.bar(x - w/2, aucs, w, label="ROC-AUC", color="#2196F3")
    b2 = ax.bar(x + w/2, aps, w, label="Avg Precision", color="#4CAF50")
    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                        f"{h:.3f}", ha="center", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15)
    ax.set_ylim(0, 1.1)
    ax.set_title("Cross-domain: ROC-AUC / AP", fontsize=14)
    ax.set_ylabel("Score")
    ax.legend()

    # in-domain reference line
    if results.get("in_domain", {}).get("roc_auc"):
        ax.axhline(results["in_domain"]["roc_auc"], color="#2196F3",
                   linestyle="--", alpha=0.5, label="In-domain AUC")

    # Clustering coverage bar chart
    ax = axes[1]
    coverages = [r.get("clustering", {}).get("coverage", 0) for r in cross]
    n_clusters = [r.get("clustering", {}).get("n_clusters", 0) for r in cross]
    bars = ax.bar(names, coverages, color="#FF9800")
    for bar, nc in zip(bars, n_clusters):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{nc} кл.", ha="center", fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_title("Cross-domain: HDBSCAN Coverage", fontsize=14)
    ax.set_ylabel("Coverage")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "v3_cross_domain_results.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Cross-domain результаты отсутствуют")

---
## 4. Сводная таблица

In [ ]:
import pandas as pd

rows = []

# In-domain
if results.get("in_domain"):
    row = {"Dataset": "IN-DOMAIN (test)", "Type": "in-domain"}
    row.update(results["in_domain"])
    if results.get("in_domain_clustering"):
        cl = results["in_domain_clustering"]
        row["hdbscan_clusters"] = cl.get("n_clusters")
        row["hdbscan_coverage"] = cl.get("coverage")
        row["hdbscan_ari"] = cl.get("ari")
        row["hdbscan_nmi"] = cl.get("nmi")
    rows.append(row)

# Cross-domain
for r in results.get("cross_domain", []):
    row = {"Dataset": r["name"], "Type": "cross-domain"}
    if "roc_auc" in r:
        row["roc_auc"] = r["roc_auc"]
    if "avg_precision" in r:
        row["avg_precision"] = r["avg_precision"]
    if "clustering" in r:
        cl = r["clustering"]
        row["hdbscan_clusters"] = cl.get("n_clusters")
        row["hdbscan_coverage"] = cl.get("coverage")
    rows.append(row)

df = pd.DataFrame(rows)
# Форматирование
for col in ["roc_auc", "avg_precision", "hdbscan_coverage", "hdbscan_ari", "hdbscan_nmi"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "—")

df

---
## 5. Распределение эмбеддингов (t-SNE)

In [ ]:
from sklearn.manifold import TSNE
from table_unifier.config import Config, EntityResolutionConfig
from table_unifier.models.entity_resolution import EntityResolutionGAT
from table_unifier.training.er_trainer import get_row_embeddings


def load_model_and_embeddings(graph, model_path, device="cpu"):
    """Загрузить модель и получить эмбеддинги."""
    er_config = EntityResolutionConfig(
        hidden_dim=128, edge_dim=128, num_gnn_layers=3,
        dropout=0.0, bidirectional=True,
        num_heads=4, attention_dropout=0.1,
    )
    er_config.row_dim = int(graph["row"].x.shape[1])
    er_config.token_dim = int(graph["token"].x.shape[1])
    er_config.col_dim = int(graph.col_embeddings.shape[1])

    model = EntityResolutionGAT(
        row_dim=er_config.row_dim, token_dim=er_config.token_dim, col_dim=er_config.col_dim,
        hidden_dim=er_config.hidden_dim, edge_dim=er_config.edge_dim, output_dim=er_config.output_dim,
        num_gnn_layers=er_config.num_gnn_layers, num_heads=er_config.num_heads,
        dropout=er_config.dropout, attention_dropout=er_config.attention_dropout,
        bidirectional=er_config.bidirectional,
    )
    state = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    return get_row_embeddings(model, graph, device=device)

In [ ]:
# t-SNE дл�� cross-domain датасетов
model_path = OUTPUT_DIR / "v3_gat_model.pt"
cross_dirs = sorted([d for d in CROSS_DIR.iterdir() if d.is_dir() and (d / "graph.pt").exists()])

if cross_dirs:
    n_plots = len(cross_dirs)
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    for ax, ds_dir in zip(axes, cross_dirs):
        name = ds_dir.name
        graph = torch.load(ds_dir / "graph.pt", weights_only=False)
        emb = load_model_and_embeddings(graph, model_path)

        # Subsample for speed
        n = min(2000, len(emb))
        idx = np.random.choice(len(emb), n, replace=False)
        emb_sub = emb[idx].numpy()

        tsne = TSNE(n_components=2, perplexity=30, random_state=42)
        proj = tsne.fit_transform(emb_sub)

        # Раскраска: HDBSCAN кластеры
        from table_unifier.evaluation.clustering import cluster_embeddings
        labels = cluster_embeddings(emb[idx], min_cluster_size=10)

        noise_mask = labels == -1
        ax.scatter(proj[noise_mask, 0], proj[noise_mask, 1],
                   c="#BDBDBD", s=8, alpha=0.3, label="noise")
        ax.scatter(proj[~noise_mask, 0], proj[~noise_mask, 1],
                   c=labels[~noise_mask], cmap="tab20", s=12, alpha=0.6)
        ax.set_title(f"{name}\n{(~noise_mask).sum()} clustered / {noise_mask.sum()} noise", fontsize=13)
        ax.set_xticks([])
        ax.set_yticks([])

        del graph, emb
        torch.cuda.empty_cache()

    plt.suptitle("t-SNE визуализация cross-domain эмбеддингов", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "v3_tsne_cross_domain.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Cross-domain графы не найдены")

---
## 6. Сохранение всех графиков

In [ ]:
from pathlib import Path

saved = list(OUTPUT_DIR.glob("v3_*.png"))
print(f"Сохранено {len(saved)} графиков в {OUTPUT_DIR}:")
for p in sorted(saved):
    print(f"  {p.name}")